## STOCK PRICE FORECASTING WITH LSTM 

In [ ]:
#Zaman Akış İhlali (Data Leakage)
#Ardışık Bağımlılığın (Sequentiality) İhmali: 
#Veri Ölçekleme (Scaling) Sorunu: Finansal veriler kriz anlarında veya ani hareketlerde çok yüksek varyans gösterir. 

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import joblib

# Finansal Zaman Serisi Verisi Uretimi
np.random.seed(42)
gun_sayisi = 1000
baslangic_fiyati = 100
fiyat_degisimleri = np.random.normal(loc=0.05, scale=1.5, size=gun_sayisi)
kapanis_fiyatlari = baslangic_fiyati + np.cumsum(fiyat_degisimleri)

df = pd.DataFrame({'Tarih': pd.date_range(start='2023-01-01', periods=gun_sayisi), 'Kapanis': kapanis_fiyatlari})

# Veri Hazirligi
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df['Kapanis'].values.reshape(-1, 1))

# Kayan Pencere
X, y = [], []
pencere_boyutu = 5
for i in range(pencere_boyutu, len(scaled_data)):
    X.append(scaled_data[i-pencere_boyutu:i, 0])
    y.append(scaled_data[i, 0])

X, y = np.array(X), np.array(y)
bolum_noktasi = int(len(X) * 0.8)
X_train, X_test = X[:bolum_noktasi], X[bolum_noktasi:]
y_train, y_test = y[:bolum_noktasi], y[bolum_noktasi:]

# Kayit
joblib.dump(scaler, 'price_scaler.pkl')
print("Model bileşenleri başarıyla oluşturuldu ve kaydedildi.")

Model bileşenleri başarıyla oluşturuldu ve kaydedildi.


In [ ]:
#Kronolojik Bölme (Time Series Split): Veriyi karıştırmadan, zaman akışına sadık kalarak (ilk %80 eğitim, son %20 test) böldük. 
# "geçmiş 5 günün fiyatı -> 6. günün tahmini" şeklinde paketleyerek derin öğrenmenin anlayacağı matris formuna dönüştürdük.

# Min-Max Normalizasyonu: Tüm finansal verileri yapay sinir ağının en kararlı çalışacağı [0, 1] aralığına normalize ettik ve tahmin sonrası inverse_transform ile gerçek finansal değere geri döndürdük.

# LSTM Derin Öğrenme Mimarisi: Zaman serilerindeki uzun vadeli trendleri ve kısa vadeli dalgalanmaları (forget & input gates) yönetebilen LSTM (Long Short-Term Memory) mimarisini